# YOLO26: детекция объектов

Загрузим предобученную модель, найдём объекты на изображении и посмотрим результат. Запускайте ячейки сверху вниз. Видео можно обработать в последней ячейке.

In [ ]:
%pip install -q "ultralytics>=8.4,<9"

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image

model = YOLO("yolo26n.pt")  # веса загрузятся автоматически

## Пример изображения

Ultralytics загрузит пример изображения. Подписи показывают класс и confidence.

In [ ]:
results = model.predict("https://ultralytics.com/images/bus.jpg", save=True, project="runs/demo", name="image", exist_ok=True)
display(Image(filename="runs/demo/image/bus.jpg"))

## Своё изображение

Выберите файл в диалоге Colab. Затем выведем изображение с результатами.

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = next(iter(uploaded))
results = model.predict(image_path, verbose=False)
results[0].save(filename="result.jpg")
display(Image(filename="result.jpg"))
files.download("result.jpg")

## Видео

Выберите видео. Обработка идёт покадрово; длинный файл может занять много времени. Результат будет без звука.

In [ ]:
import cv2
import subprocess
import time

uploaded_video = files.upload()
video_path = next(iter(uploaded_video))
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
assert cap.isOpened() and fps > 0, "Не удалось открыть видео"
writer = None
frames = 0
start = time.perf_counter()
while True:
    ok, frame = cap.read()
    if not ok:
        break
    annotated = model.predict(frame, verbose=False)[0].plot()
    if writer is None:
        h, w = annotated.shape[:2]
        writer = cv2.VideoWriter("result_video.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        assert writer.isOpened(), "Не удалось создать видео"
    writer.write(annotated)
    frames += 1
cap.release()
if writer is not None:
    writer.release()
assert frames, "Не удалось прочитать кадры"
print(f"Кадров: {frames}, среднее время: {(time.perf_counter()-start)*1000/frames:.1f} мс/кадр")
files.download("result_video.mp4")